# Credit Score Classification

Predict a customer's credit score (**Poor / Standard / Good**) from financial and personal attributes.

**Data:** `train.csv` (100,000 rows, labeled) and `test.csv` (50,000 rows, **no labels**).
All training and evaluation use `train.csv`; `test.csv` is used only for the final predictions.

**How to run (Google Colab):** Runtime -> Run all. When asked, upload `train.csv` and `test.csv`.
The last cell builds the final report and a zip of everything to upload to GitHub.

## Step 1: Setup and load data

In [ ]:
%pip install -q xgboost shap python-docx
import os, time, shutil, warnings, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
sns.set_style("whitegrid")
for d in ["figures", "outputs", "reports"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
# Upload train.csv and test.csv if they are not already in the Colab session
if not (os.path.exists("train.csv") and os.path.exists("test.csv")):
    from google.colab import files
    files.upload()

train = pd.read_csv("train.csv", low_memory=False)
test = pd.read_csv("test.csv", low_memory=False)
print("Train:", train.shape, "| Test:", test.shape)
print("Columns only in train:", set(train.columns) - set(test.columns))
train.head()

## Step 2: Explore the raw data

In [ ]:
train.info()
print("\nDuplicate rows:", train.duplicated().sum())
miss = train.isna().sum()
display(pd.DataFrame({"missing": miss, "pct": (miss / len(train) * 100).round(2)})[miss > 0])

print(train["Credit_Score"].value_counts(normalize=True).round(3))
print("\nCustomers:", train["Customer_ID"].nunique())
print(train.groupby("Customer_ID").size().value_counts())
print("Train months:", train["Month"].unique())
print("Test months:", test["Month"].unique())

In [ ]:
for c in ["Age", "Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment",
          "Changed_Credit_Limit", "Outstanding_Debt", "Amount_invested_monthly", "Monthly_Balance"]:
    print(c, "->", train[c].sample(6, random_state=1).tolist())
for c in ["Occupation", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]:
    print(f"\n--- {c} ---")
    print(train[c].value_counts(dropna=False))

## Step 3: Clean the data
Junk values become NaN, impossible values become NaN, then gaps are filled from the same customer's other months.

In [ ]:
JUNK = ["_______", "!@9#%8", "_", "#F%$D@*&8", "nan"]
num_cols = ["Age", "Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment",
            "Changed_Credit_Limit", "Outstanding_Debt", "Amount_invested_monthly", "Monthly_Balance"]
bounds = {"Age": (14, 100), "Num_Bank_Accounts": (0, 11), "Num_Credit_Card": (0, 11),
          "Interest_Rate": (1, 34), "Num_of_Loan": (0, 9),
          "Num_of_Delayed_Payment": (0, 28), "Num_Credit_Inquiries": (0, 17)}
mode_cols = ["Age", "Occupation", "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts",
             "Num_Credit_Card", "Interest_Rate", "Num_of_Loan", "Credit_Mix", "Payment_Behaviour"]
median_cols = ["Num_of_Delayed_Payment", "Changed_Credit_Limit", "Num_Credit_Inquiries",
               "Outstanding_Debt", "Amount_invested_monthly", "Monthly_Balance",
               "Credit_History_Months", "Delay_from_due_date", "Total_EMI_per_month"]

def to_months(s):
    if pd.isna(s):
        return np.nan
    p = str(s).split()
    return int(p[0]) * 12 + int(p[3])

def fill_mode(s):
    m = s.mode()
    return s.fillna(m.iloc[0]) if len(m) else s

def clean(df):
    df = df.copy().replace(JUNK, np.nan)
    for c in num_cols:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace("_", "", regex=False), errors="coerce")
    df["Credit_History_Months"] = df["Credit_History_Age"].apply(to_months)
    df = df.drop(columns=["Credit_History_Age"])
    for c, (lo, hi) in bounds.items():
        df.loc[(df[c] < lo) | (df[c] > hi), c] = np.nan
    df.loc[df["Monthly_Balance"] < -1e6, "Monthly_Balance"] = np.nan
    df.loc[df["Total_EMI_per_month"] > 1500, "Total_EMI_per_month"] = np.nan
    df.loc[df["Amount_invested_monthly"] > 5000, "Amount_invested_monthly"] = np.nan
    for c in mode_cols:
        df[c] = df.groupby("Customer_ID")[c].transform(fill_mode)
    for c in median_cols:
        df[c] = df.groupby("Customer_ID")[c].transform(lambda s: s.fillna(s.median()))
    df["Type_of_Loan"] = df["Type_of_Loan"].fillna("No Loan")
    return df

def fill_leftover(df, ref):
    for c in df.columns:
        if df[c].isna().any():
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = df[c].fillna(ref[c].mode().iloc[0])
            else:
                df[c] = df[c].fillna(ref[c].median())
    return df

train_c = clean(train)
test_c = clean(test)
train_ref = train_c.copy()
train_c = fill_leftover(train_c, train_ref)
test_c = fill_leftover(test_c, train_ref)   # test uses train statistics only
print("Remaining NaNs  train:", train_c.isna().sum().sum(), "| test:", test_c.isna().sum().sum())
train_c.describe().T

In [ ]:
for c in ["Occupation", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]:
    print(f"\n--- {c} ---")
    print(train_c[c].value_counts())

## Step 4: Feature engineering and encoding
`Month` is dropped: train covers Jan-Aug while test covers different months, so it would not generalize.

In [ ]:
loan_types = ["Auto Loan", "Credit-Builder Loan", "Debt Consolidation Loan", "Home Equity Loan",
              "Mortgage Loan", "Payday Loan", "Personal Loan", "Student Loan"]

def engineer(df):
    df = df.copy()
    df["Loan_Count"] = df["Type_of_Loan"].apply(lambda s: 0 if s == "No Loan" else len(str(s).split(",")))
    for lt in loan_types:
        df["Has_" + lt.replace(" ", "_").replace("-", "_")] = df["Type_of_Loan"].str.contains(lt, regex=False).astype(int)
    df["Debt_to_Income"] = df["Outstanding_Debt"] / (df["Annual_Income"] + 1)
    df["EMI_to_Salary"] = df["Total_EMI_per_month"] / (df["Monthly_Inhand_Salary"] + 1)
    df["Invest_to_Salary"] = df["Amount_invested_monthly"] / (df["Monthly_Inhand_Salary"] + 1)
    df["Balance_to_Salary"] = df["Monthly_Balance"] / (df["Monthly_Inhand_Salary"] + 1)
    return df

train_f = engineer(train_c)
test_f = engineer(test_c)
eda = train_f.copy()                       # readable copy (text labels) for Step 5

test_ids = test_f["ID"].copy()
groups = train_f["Customer_ID"].copy()
drop_cols = ["ID", "Customer_ID", "Name", "SSN", "Month", "Type_of_Loan"]
train_f = train_f.drop(columns=drop_cols)
test_f = test_f.drop(columns=drop_cols)

credit_mix_map = {"Bad": 0, "Standard": 1, "Good": 2}
for df in (train_f, test_f):
    df["Credit_Mix"] = df["Credit_Mix"].map(credit_mix_map)

cat_cols = ["Occupation", "Payment_of_Min_Amount", "Payment_Behaviour"]
train_f = pd.get_dummies(train_f, columns=cat_cols)
test_f = pd.get_dummies(test_f, columns=cat_cols)
feature_cols = [c for c in train_f.columns if c != "Credit_Score"]
test_f = test_f.reindex(columns=feature_cols, fill_value=0)

target_map = {"Poor": 0, "Standard": 1, "Good": 2}
y = train_f["Credit_Score"].map(target_map)
X = train_f[feature_cols].astype(float)
X_test = test_f[feature_cols].astype(float)

print("X:", X.shape, "| y:", y.shape, "| X_test:", X_test.shape)
print("NaNs in X:", X.isna().sum().sum(), "| NaNs in X_test:", X_test.isna().sum().sum())
print("Same columns:", list(X.columns) == list(X_test.columns))

## Step 5: EDA visualizations

In [ ]:
order = ["Poor", "Standard", "Good"]
palette = {"Poor": "#e74c3c", "Standard": "#f1c40f", "Good": "#2ecc71"}

def save(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

# 1. Target distribution
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
counts = eda["Credit_Score"].value_counts()[order]
ax[0].pie(counts, labels=order, autopct="%1.1f%%", colors=[palette[o] for o in order], startangle=90)
ax[0].set_title("Credit Score Share")
sns.countplot(data=eda, x="Credit_Score", order=order, hue="Credit_Score", palette=palette, legend=False, ax=ax[1])
ax[1].set_title("Credit Score Count")
save("01_target_distribution")

# 2. Key numeric features by score
key_num = ["Annual_Income", "Outstanding_Debt", "Interest_Rate", "Delay_from_due_date",
           "Num_of_Delayed_Payment", "Credit_Utilization_Ratio", "Credit_History_Months", "Num_Credit_Inquiries"]
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, c in zip(axes.ravel(), key_num):
    sns.boxplot(data=eda, x="Credit_Score", y=c, order=order, hue="Credit_Score", palette=palette,
                legend=False, showfliers=False, ax=ax)
    ax.set_title(c)
save("02_numeric_boxplots")

# 3. KDE distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, c in zip(axes.ravel(), ["Age", "Annual_Income", "Outstanding_Debt", "Interest_Rate",
                                "Credit_History_Months", "Monthly_Balance"]):
    for s in order:
        sns.kdeplot(eda.loc[eda["Credit_Score"] == s, c], label=s, color=palette[s], fill=True, alpha=0.25, ax=ax)
    ax.set_title(c)
    ax.legend()
save("03_kde_distributions")

In [ ]:
# 4. Categorical features vs score
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, c in zip(axes, ["Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]):
    pct = pd.crosstab(eda[c], eda["Credit_Score"], normalize="index")[order] * 100
    pct.plot(kind="bar", stacked=True, color=[palette[o] for o in order], ax=ax)
    ax.set_title(f"{c} vs Credit Score (%)")
    ax.set_ylabel("%")
    ax.tick_params(axis="x", rotation=45)
save("04_categorical_vs_score")

# 5. Occupation vs score
pct = (pd.crosstab(eda["Occupation"], eda["Credit_Score"], normalize="index")[order] * 100).sort_values("Good")
pct.plot(kind="barh", stacked=True, figsize=(10, 7), color=[palette[o] for o in order])
plt.title("Credit Score Mix by Occupation (%)")
plt.xlabel("%")
save("05_occupation_vs_score")

# 6. Loans and accounts
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, c in zip(axes, ["Loan_Count", "Num_Bank_Accounts", "Num_Credit_Card"]):
    sns.barplot(data=eda, x="Credit_Score", y=c, order=order, hue="Credit_Score", palette=palette,
                legend=False, errorbar=None, ax=ax)
    ax.set_title(f"Average {c}")
save("06_loans_accounts")

# 7. Correlation heatmap
corr_cols = ["Age", "Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts", "Num_Credit_Card",
             "Interest_Rate", "Num_of_Loan", "Delay_from_due_date", "Num_of_Delayed_Payment",
             "Changed_Credit_Limit", "Num_Credit_Inquiries", "Outstanding_Debt", "Credit_Utilization_Ratio",
             "Credit_History_Months", "Total_EMI_per_month", "Amount_invested_monthly", "Monthly_Balance",
             "Debt_to_Income", "EMI_to_Salary"]
plt.figure(figsize=(14, 11))
sns.heatmap(eda[corr_cols].corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7})
plt.title("Correlation Heatmap")
save("07_correlation_heatmap")

# 8. Correlation with target
binary = [c for c in X.columns if X[c].nunique() <= 2]
corr_target = X.drop(columns=binary).corrwith(y).sort_values()
corr_target.plot(kind="barh", figsize=(9, 8), color="steelblue")
plt.title("Correlation of Numeric Features with Credit_Score (0=Poor, 2=Good)")
save("08_corr_with_target")

In [ ]:
# Interactive Plotly charts (display only)
samp = eda.sample(5000, random_state=42)
px.scatter(samp, x="Annual_Income", y="Outstanding_Debt", color="Credit_Score", color_discrete_map=palette,
           category_orders={"Credit_Score": order}, opacity=0.6, title="Annual Income vs Outstanding Debt").show()
px.box(eda, x="Credit_Score", y="Interest_Rate", color="Credit_Score", color_discrete_map=palette,
       category_orders={"Credit_Score": order}, title="Interest Rate by Credit Score").show()
occ = eda.groupby(["Occupation", "Credit_Score"]).size().reset_index(name="count")
px.bar(occ, x="Occupation", y="count", color="Credit_Score", color_discrete_map=palette, barmode="group",
       category_orders={"Credit_Score": order}, title="Credit Score Count by Occupation").show()

## Step 6: Split train.csv by customer (prevents leakage)
Each customer has 8 similar monthly rows, so all rows of a customer must stay in the same set.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

X = X.reset_index(drop=True); y = y.reset_index(drop=True); groups = groups.reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf.split(X, y, groups))
X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

print("Train:", X_train.shape, "| Validation:", X_val.shape)
print("Customers in both sets (must be 0):", len(set(groups.iloc[train_idx]) & set(groups.iloc[val_idx])))
dist = pd.DataFrame({"Full": y.value_counts(normalize=True), "Train": y_train.value_counts(normalize=True),
                     "Validation": y_val.value_counts(normalize=True)}).round(3)
dist.index = dist.index.map({0: "Poor", 1: "Standard", 2: "Good"})
dist

## Step 7: Train models
Logistic Regression (baseline, scaled inside a pipeline), Random Forest and XGBoost.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),
                                     ("clf", LogisticRegression(max_iter=1000, random_state=42))]),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.1, subsample=0.8,
                             colsample_bytree=0.8, objective="multi:softprob", eval_metric="mlogloss",
                             n_jobs=-1, random_state=42),
}
fitted, rows = {}, []
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    secs = time.time() - t0
    fitted[name] = model
    rows.append({"Model": name,
                 "Train accuracy": accuracy_score(y_train, model.predict(X_train)),
                 "Validation accuracy": accuracy_score(y_val, model.predict(X_val)),
                 "Validation macro F1": f1_score(y_val, model.predict(X_val), average="macro"),
                 "Train time (s)": round(secs, 1)})
    print(f"{name} done in {secs:.1f}s")
train_results = pd.DataFrame(rows).round(4)
train_results["Gap"] = (train_results["Train accuracy"] - train_results["Validation accuracy"]).round(4)
train_results

## Step 8: Evaluate and compare

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
names = ["Poor", "Standard", "Good"]
preds = {n: m.predict(X_val) for n, m in fitted.items()}
for n, p in preds.items():
    print(f"\n{'=' * 55}\n{n}\n{'=' * 55}")
    print(classification_report(y_val, p, target_names=names, digits=3))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, (n, p) in enumerate(preds.items()):
    cm = confusion_matrix(y_val, p, labels=[0, 1, 2])
    cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=names, yticklabels=names, ax=axes[0, i])
    axes[0, i].set_title(f"{n} (counts)"); axes[0, i].set_xlabel("Predicted"); axes[0, i].set_ylabel("Actual")
    sns.heatmap(cm_pct, annot=True, fmt=".1f", cmap="Greens", cbar=False, xticklabels=names, yticklabels=names, ax=axes[1, i])
    axes[1, i].set_title(f"{n} (% of actual class)"); axes[1, i].set_xlabel("Predicted"); axes[1, i].set_ylabel("Actual")
save("09_confusion_matrices")

In [ ]:
rows = []
for n, p in preds.items():
    rep = classification_report(y_val, p, target_names=names, output_dict=True)
    rows.append({"Model": n, "Accuracy": rep["accuracy"], "Macro F1": rep["macro avg"]["f1-score"],
                 "Weighted F1": rep["weighted avg"]["f1-score"], "Poor F1": rep["Poor"]["f1-score"],
                 "Standard F1": rep["Standard"]["f1-score"], "Good F1": rep["Good"]["f1-score"]})
summary = pd.DataFrame(rows).set_index("Model").sort_values("Macro F1", ascending=False).round(4)
display(summary)

summary[["Accuracy", "Macro F1", "Weighted F1"]].plot(kind="bar", figsize=(10, 5), colormap="viridis")
plt.title("Model Comparison on Validation Set"); plt.ylabel("Score"); plt.ylim(0, 1); plt.xticks(rotation=0)
save("10_model_comparison")

summary[["Poor F1", "Standard F1", "Good F1"]].plot(kind="bar", figsize=(10, 5), color=[palette[o] for o in order])
plt.title("Per-Class F1 by Model"); plt.ylabel("F1"); plt.ylim(0, 1); plt.xticks(rotation=0)
save("11_per_class_f1")

best_name = summary["Macro F1"].idxmax()
print("Best model (by macro F1):", best_name)

## Step 9: Feature importance

In [ ]:
imp = pd.DataFrame({"Random Forest": fitted["Random Forest"].feature_importances_,
                    "XGBoost": fitted["XGBoost"].feature_importances_}, index=X.columns)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, n in zip(axes, ["Random Forest", "XGBoost"]):
    imp[n].sort_values().tail(15).plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(f"{n}: Top 15 Features"); ax.set_xlabel("Importance")
save("12_feature_importance")

# Permutation importance on unseen validation data (sampled for speed)
from sklearn.inspection import permutation_importance
Xv = X_val.sample(min(5000, len(X_val)), random_state=42)
yv = y_val.loc[Xv.index]
perm = permutation_importance(fitted[best_name], Xv, yv, n_repeats=3, random_state=42, scoring="accuracy", n_jobs=1)
perm_df = pd.DataFrame({"Feature": X.columns, "Importance": perm.importances_mean,
                        "Std": perm.importances_std}).sort_values("Importance", ascending=False)
display(perm_df.head(15))
top = perm_df.head(15).iloc[::-1]
plt.figure(figsize=(9, 7))
plt.barh(top["Feature"], top["Importance"], xerr=top["Std"], color="darkorange")
plt.title(f"Permutation Importance ({best_name}): Top 15"); plt.xlabel("Drop in accuracy when shuffled")
save("13_permutation_importance")

In [ ]:
def base_name(c):
    for p in ["Occupation_", "Payment_of_Min_Amount_", "Payment_Behaviour_"]:
        if c.startswith(p):
            return p[:-1]
    return c

grouped = imp.assign(group=imp.index.map(base_name)).groupby("group").sum()
grouped["Average"] = grouped.mean(axis=1)
grouped = grouped.sort_values("Average", ascending=False)
grouped["Average"].head(15).iloc[::-1].plot(kind="barh", figsize=(9, 7), color="seagreen")
plt.title("Feature Importance (one-hot columns combined)"); plt.xlabel("Average importance (RF and XGBoost)")
save("14_grouped_importance")
grouped.head(15)

In [ ]:
# Optional: SHAP summary (skipped automatically if it fails)
try:
    import shap
    Xs = X_val.sample(2000, random_state=42)
    sv = shap.TreeExplainer(fitted["XGBoost"]).shap_values(Xs)
    shap.summary_plot(sv, Xs, class_names=names, max_display=15, show=False)
    plt.savefig("figures/16_shap_summary.png", dpi=130, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("SHAP skipped:", e)

## Step 10: Predict on test.csv
Retrain the best model on all of train.csv, then score test.csv. test.csv has no labels, so no accuracy can be computed on it.

In [ ]:
from sklearn.base import clone
final_model = clone(models[best_name]).fit(X, y)

assert list(X_test.columns) == list(X.columns) and X_test.isna().sum().sum() == 0
inv_map = {0: "Poor", 1: "Standard", 2: "Good"}
test_pred = final_model.predict(X_test)
test_labels = pd.Series(test_pred).map(inv_map)
proba = final_model.predict_proba(X_test)

predictions = pd.DataFrame({"ID": test_ids.values, "Predicted_Credit_Score": test_labels.values,
                            "Prob_Poor": proba[:, 0].round(4), "Prob_Standard": proba[:, 1].round(4),
                            "Prob_Good": proba[:, 2].round(4)})
predictions.to_csv("outputs/predictions.csv", index=False)
print(predictions.shape)

compare = pd.DataFrame({"Train (actual)": train["Credit_Score"].value_counts(normalize=True),
                        "Test (predicted)": test_labels.value_counts(normalize=True)}).round(3).loc[order]
display(compare)
compare.plot(kind="bar", figsize=(8, 4), color=["#3498db", "#e67e22"])
plt.title("Score Distribution: Train (actual) vs Test (predicted)"); plt.ylabel("Share"); plt.xticks(rotation=0)
save("15_train_vs_test_distribution")
predictions.head(10)

## Step 11: Build the final report, README and upload package
This cell writes everything using your real results and creates `upload_package.zip`.

In [ ]:
from docx import Document
from docx.shared import Inches

summary.to_csv("outputs/evaluation_summary.csv")
train_results.to_csv("outputs/train_vs_validation.csv", index=False)
imp.to_csv("outputs/feature_importance.csv")
perm_df.to_csv("outputs/permutation_importance.csv", index=False)

best = summary.loc[best_name]
top_feats = perm_df.head(5)["Feature"].tolist()
top_grouped = grouped.head(5).index.tolist()
medians = eda.groupby("Credit_Score")[["Interest_Rate", "Outstanding_Debt", "Delay_from_due_date",
                                        "Num_of_Delayed_Payment", "Credit_History_Months",
                                        "Credit_Utilization_Ratio"]].median().loc[order].round(2)

# ---------------- Report (.docx) ----------------
doc = Document()
def para(t): doc.add_paragraph(t)
def add_table(df, index=True):
    d = df.reset_index() if index else df
    t = doc.add_table(rows=1, cols=len(d.columns)); t.style = "Light Grid Accent 1"
    for i, c in enumerate(d.columns):
        t.rows[0].cells[i].text = str(c)
    for _, r in d.iterrows():
        cells_ = t.add_row().cells
        for i, v in enumerate(r):
            cells_[i].text = f"{v:.4f}" if isinstance(v, float) else str(v)
def add_fig(f, cap):
    if os.path.exists(f"figures/{f}.png"):
        doc.add_picture(f"figures/{f}.png", width=Inches(6)); para(cap)

doc.add_heading("Credit Score Classification: Project Report", 0)

doc.add_heading("1. Introduction", 1)
para("The goal of this project is to predict a customer's credit score (Poor, Standard or Good) from financial and "
     "personal attributes, so that credit assessment can be automated, consistent and faster. The data has "
     f"{len(train):,} labeled training rows ({train['Customer_ID'].nunique():,} customers, 8 monthly records each) and "
     f"{len(test):,} unlabeled test rows.")

doc.add_heading("2. Data cleaning", 1)
para("The raw data contained stray characters (for example trailing underscores), placeholder strings, numbers stored as "
     "text, impossible values (negative ages, extreme counts of accounts or loans) and missing values. Junk and impossible "
     "values were converted to missing, numeric columns were converted to numbers, and Credit_History_Age was converted to "
     "months. Missing values were then filled using the same customer's other months (mode for categorical and "
     "fixed attributes, median for varying amounts). Any remaining gaps were filled with training-set statistics.")

doc.add_heading("3. Feature engineering", 1)
para("New features: loan count, one flag per loan type, debt-to-income, EMI-to-salary, investment-to-salary and "
     "balance-to-salary ratios. Identifier columns (ID, Customer_ID, Name, SSN) and Month were dropped. Credit_Mix was "
     "encoded as an ordered value (Bad < Standard < Good); Occupation, Payment_of_Min_Amount and Payment_Behaviour were one-hot "
     "encoded. The target was encoded as Poor=0, Standard=1, Good=2.")

doc.add_heading("4. Exploratory data analysis", 1)
para("Median values of key features by credit score class:")
add_table(medians)
add_fig("01_target_distribution", "Figure 1: Target distribution.")
add_fig("02_numeric_boxplots", "Figure 2: Key numeric features by credit score.")
add_fig("04_categorical_vs_score", "Figure 3: Categorical features versus credit score.")
add_fig("05_occupation_vs_score", "Figure 4: Credit score mix by occupation.")
add_fig("07_correlation_heatmap", "Figure 5: Correlation heatmap.")
add_fig("08_corr_with_target", "Figure 6: Correlation of features with the credit score.")

doc.add_heading("5. Methodology", 1)
para("train.csv was split 80/20 by customer (StratifiedGroupKFold on Customer_ID) so that no customer appears in both "
     "the training and validation sets; a random row split would leak a customer's near-identical monthly records and "
     "inflate scores. Three models were trained: Logistic Regression (baseline, with standard scaling inside a pipeline), "
     "Random Forest and XGBoost. The tree models need no scaling. Because the classes are imbalanced, the best model was "
     "chosen by macro F1.")

doc.add_heading("6. Model comparison", 1)
para("Results on the held-out validation set:")
add_table(summary)
para(f"Best model: {best_name} (accuracy {best['Accuracy']:.4f}, macro F1 {best['Macro F1']:.4f}).")
para("Training versus validation accuracy (a large gap indicates overfitting):")
add_table(train_results, index=False)
add_fig("10_model_comparison", "Figure 7: Model comparison.")
add_fig("11_per_class_f1", "Figure 8: Per-class F1.")
add_fig("09_confusion_matrices", "Figure 9: Confusion matrices.")

doc.add_heading("7. Feature importance", 1)
para(f"The five most important features by permutation importance ({best_name}) were: " + ", ".join(top_feats) + ". "
     "With one-hot columns combined, the top factors by average built-in importance were: " + ", ".join(top_grouped) + ".")
add_table(perm_df.head(10)[["Feature", "Importance"]], index=False)
add_fig("13_permutation_importance", "Figure 10: Permutation importance.")
add_fig("14_grouped_importance", "Figure 11: Grouped feature importance.")
add_fig("12_feature_importance", "Figure 12: Built-in importance (Random Forest and XGBoost).")

doc.add_heading("8. Predictions on test.csv", 1)
para(f"The best model was retrained on all of train.csv and used to predict the {len(test):,} test rows "
     "(outputs/predictions.csv). Predicted class shares:")
add_table(compare)
para("test.csv has no Credit_Score column, so accuracy cannot be measured on it. All reported metrics come from the "
     "customer-grouped validation split of train.csv.")
add_fig("15_train_vs_test_distribution", "Figure 13: Train (actual) versus test (predicted) distribution.")

doc.add_heading("9. Limitations and future work", 1)
para("No labels exist for test.csv; hyperparameters were not extensively tuned; the data is partly synthetic and "
     "heavily noisy; the Standard class is the largest and hardest to separate. Future work: hyperparameter search, "
     "class weighting, additional models (LightGBM), and probability calibration.")

doc.add_heading("10. Conclusion", 1)
para(f"{best_name} performed best on the validation set (accuracy {best['Accuracy']:.4f}, macro F1 {best['Macro F1']:.4f}). "
     f"The most influential factors were {', '.join(top_feats[:3])}, which links back to the patterns seen in the EDA.")
doc.save("reports/final_report.docx")

# ---------------- README with real numbers ----------------
md_table = summary[["Accuracy", "Macro F1"]].reset_index().to_markdown(index=False)
readme = f"""# Credit Score Classification

Machine learning model that predicts a customer's credit score (Poor / Standard / Good) from financial and personal attributes.

## Dataset
Kaggle "Credit Score Classification". `train.csv` has {len(train):,} labeled rows and `test.csv` has {len(test):,} rows without labels. Each customer has 8 monthly records.

## Approach
1. Data exploration
2. Cleaning (junk values, impossible values, per-customer imputation)
3. Feature engineering (loan counts, loan-type flags, financial ratios)
4. EDA visualizations (Matplotlib, Seaborn, Plotly)
5. Customer-grouped 80/20 split to avoid leakage
6. Models: Logistic Regression, Random Forest, XGBoost
7. Evaluation: accuracy, classification report, confusion matrix, macro F1
8. Feature importance: built-in, permutation, SHAP
9. Predictions on test.csv

## Results (validation set)
{md_table}

Best model: **{best_name}**

## Top features (permutation importance)
""" + "\n".join(f"{i+1}. {f}" for i, f in enumerate(top_feats)) + f"""

## Repository contents
- `notebooks/credit_score_classification.ipynb`: full analysis and model code
- `reports/final_report.docx`: final report
- `outputs/`: predictions, evaluation summary, feature importance tables
- `figures/`: all charts

## Note
`test.csv` has no `Credit_Score`, so metrics come from a customer-grouped validation split of `train.csv`. `test.csv` is used only for final predictions.

## How to run
```
pip install -r requirements.txt
```
Open the notebook in Jupyter or Google Colab, place `train.csv` and `test.csv` next to it, and run all cells.
"""
open("README.md", "w").write(readme)

open("requirements.txt", "w").write("pandas\nnumpy\nmatplotlib\nseaborn\nplotly\nscikit-learn\nxgboost\nshap\njoblib\npython-docx\ntabulate\n")
open(".gitignore", "w").write(".ipynb_checkpoints/\n__pycache__/\n*.pkl\n.DS_Store\n")

# ---------------- Zip for GitHub ----------------
pkg = "package"
shutil.rmtree(pkg, ignore_errors=True)
for d in ["figures", "outputs", "reports"]:
    shutil.copytree(d, f"{pkg}/{d}")
for f in ["README.md", "requirements.txt", ".gitignore"]:
    shutil.copy(f, f"{pkg}/{f}")
os.makedirs(f"{pkg}/notebooks", exist_ok=True)
shutil.make_archive("upload_package", "zip", pkg)
print("Done. Download upload_package.zip (left panel -> Files, or run the next cell).")

In [ ]:
from google.colab import files
files.download("upload_package.zip")
files.download("reports/final_report.docx")